In [0]:
CREATE SCHEMA IF NOT EXISTS workspace.myra_invest;

USE workspace.myra_invest;

CREATE TABLE IF NOT EXISTS workspace.myra_invest.bronze_stock_news (

    symbol STRING,
    companyName STRING,
    headline STRING,
    summary STRING,
    publisher STRING,
    publishedAt TIMESTAMP,
    url STRING,
    source STRING,
    ingestionTimestamp TIMESTAMP,
    ingestionDate DATE

)
USING DELTA;

SHOW TABLES IN workspace.myra_invest;

SELECT COUNT(*) AS total_news
FROM workspace.myra_invest.bronze_stock_news;


In [0]:
%python

from pyspark.sql.functions import col, to_timestamp, to_date

news_df = (
    spark.read
        .option("header", True)
        .option("multiLine", True)
        .option("escape", '"')
        .csv("/Volumes/workspace/myra_invest/news_files/bronze_stock_news.csv")
        .withColumn("publishedAt", to_timestamp(col("publishedAt")))
        .withColumn("ingestionTimestamp", to_timestamp(col("ingestionTimestamp")))
        .withColumn("ingestionDate", to_date(col("ingestionTimestamp")))
)

display(news_df)

news_df.printSchema()

(
    news_df.write
        .mode("append")
        .format("delta")
        .saveAsTable("workspace.myra_invest.bronze_stock_news")
)

print("✅ Bronze News table loaded successfully.")